## Assignment A4 - Simulated Annealing for TSP

### 1) Goal
Implement and evaluate **Simulated Annealing (SA)** for the Traveling Salesman Problem (TSP).

### 2) SA Idea (minimization)
At each step we move from current route $c$ to a neighbor $x$.

- If $f(x) < f(c)$, accept the move.
- Otherwise, accept with probability:

$$
P(\text{accept}) = e^{-\frac{f(x)-f(c)}{T}}
$$

Temperature update:

$$
T_{k+1} = \alpha \cdot T_k, \quad \alpha \in (0,1)
$$

Stop when $T < T_{min}$ or max iterations reached.

In [1]:
import math
import random
import time
from statistics import mean


def read(filename):
    """Read TSPLIB EUC_2D file and return name, dimension, and ordered coordinates."""
    name = None
    dimension = None
    in_coord_section = False
    coords_by_id = {}

    with open(filename, "r") as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line:
                continue

            if line.startswith("NAME"):
                name = line.split(":", 1)[1].strip() if ":" in line else line.split()[-1]
            elif line.startswith("DIMENSION"):
                rhs = line.split(":", 1)[1].strip() if ":" in line else line.split()[-1]
                dimension = int(rhs)
            elif line == "NODE_COORD_SECTION":
                in_coord_section = True
            elif line == "EOF":
                break
            elif in_coord_section:
                parts = line.split()
                if len(parts) >= 3:
                    city_id = int(parts[0])
                    x = float(parts[1])
                    y = float(parts[2])
                    coords_by_id[city_id] = (x, y)

    ordered_ids = sorted(coords_by_id.keys())
    coords = [coords_by_id[i] for i in ordered_ids]

    if dimension is None:
        dimension = len(coords)

    return {
        "name": name if name else filename,
        "dimension": dimension,
        "coords": coords,
    }


In [2]:
def euclidian_dist(a, b):
    return math.hypot(a[0] - b[0], a[1] - b[1])


def total_dist(route, coords):
    total = 0.0
    n = len(route)
    for i in range(n):
        c1 = coords[route[i]]
        c2 = coords[route[(i + 1) % n]]
        total += euclidian_dist(c1, c2)
    return total


def totat_dist(route, coords):
    return total_dist(route, coords)


def greedy_route(coords, start=0):
    n = len(coords)
    unvisited = set(range(n))
    route = [start]
    unvisited.remove(start)
    current = start

    while unvisited:
        next_city = min(unvisited, key=lambda c: euclidian_dist(coords[current], coords[c]))
        route.append(next_city)
        unvisited.remove(next_city)
        current = next_city

    return route


def random_route(n, start=0):
    route = list(range(n))
    middle = route[:]
    middle.remove(start)
    random.shuffle(middle)
    return [start] + middle


def random_neighbor(route):
    """2-opt random neighbor, keeping city 0 fixed at index 0."""
    n = len(route)
    if n <= 3:
        return route[:]

    i = random.randint(1, n - 2)
    j = random.randint(i + 1, n - 1)

    candidate = route[:]
    candidate[i : j + 1] = reversed(candidate[i : j + 1])
    return candidate


def random_2opt_indices(n):
    i = random.randint(1, n - 2)
    j = random.randint(i + 1, n - 1)
    return i, j


def two_opt_delta(route, coords, i, j):
    """Cost change for reversing route[i:j+1]."""
    n = len(route)
    a = route[i - 1]
    b = route[i]
    c = route[j]
    d = route[(j + 1) % n]

    old_cost = euclidian_dist(coords[a], coords[b]) + euclidian_dist(coords[c], coords[d])
    new_cost = euclidian_dist(coords[a], coords[c]) + euclidian_dist(coords[b], coords[d])
    return new_cost - old_cost


def apply_2opt(route, i, j):
    candidate = route[:]
    candidate[i : j + 1] = reversed(candidate[i : j + 1])
    return candidate

In [3]:
def run(
    coords,
    T_init=6000,
    alpha=0.995,
    T_min=1e-4,
    nr_iter=12000,
    seed=None,
    init_mode="greedy",
    neighbor_samples=25,
):
    """Simulated Annealing for TSP using sampled 2-opt neighborhood."""
    if seed is not None:
        random.seed(seed)

    n = len(coords)
    if init_mode == "greedy":
        current = greedy_route(coords, start=0)
    else:
        current = random_route(n, start=0)

    current_value = total_dist(current, coords)
    best = current[:]
    best_value = current_value

    T = float(T_init)
    accepted_moves = 0
    history = []

    for it in range(nr_iter):
        if T < T_min:
            break

        best_i = None
        best_j = None
        best_delta = float("inf")

        for _ in range(neighbor_samples):
            i, j = random_2opt_indices(n)
            delta = two_opt_delta(current, coords, i, j)
            if delta < best_delta:
                best_delta = delta
                best_i = i
                best_j = j

        if best_i is None:
            T *= alpha
            continue

        if best_delta < 0 or random.random() < math.exp(-best_delta / T):
            current = apply_2opt(current, best_i, best_j)
            current_value += best_delta
            accepted_moves += 1

            if current_value < best_value:
                best = current[:]
                best_value = current_value

        if it % 200 == 0:
            history.append((it, best_value, current_value, T))

        T *= alpha

    return {
        "best_route": best,
        "best_value": best_value,
        "accepted_moves": accepted_moves,
        "history": history,
    }

In [4]:
def run_sa_tsp_experiments(instance_paths, parameter_grid, num_runs=3, output_filename="results_sa_tsp.txt"):
    all_rows = []

    with open(output_filename, "w") as f:
        f.write("--- Simulated Annealing TSP Results ---\n")
        f.write(f"Runs per parameter setting: {num_runs}\n")

        for instance_path in instance_paths:
            tsp = read(instance_path)
            coords = tsp["coords"]
            n = tsp["dimension"]

            print(f"\n[Instance] {tsp['name']} (n={n})")
            f.write("\n" + "=" * 72 + "\n")
            f.write(f"Instance: {tsp['name']} | n={n}\n")
            f.write("=" * 72 + "\n")

            greedy = greedy_route(coords, start=0)
            greedy_value = total_dist(greedy, coords)
            print(f"  Greedy baseline: {greedy_value:.2f}")
            f.write(f"Greedy baseline length: {greedy_value:.2f}\n")

            for idx, params in enumerate(parameter_grid, start=1):
                T_init = params["T_init"]
                alpha = params["alpha"]
                T_min = params["T_min"]
                nr_iter = params["nr_iter"]
                neighbor_samples = params.get("neighbor_samples", 25)
                init_mode = params.get("init_mode", "greedy")

                lengths = []
                times = []

                print(
                    f"  [Setting {idx}/{len(parameter_grid)}] "
                    f"T_init={T_init}, alpha={alpha}, T_min={T_min}, iter={nr_iter}, "
                    f"samples={neighbor_samples}, init={init_mode}"
                )
                f.write(
                    f"\nParams: T_init={T_init}, alpha={alpha}, T_min={T_min}, nr_iter={nr_iter}, "
                    f"neighbor_samples={neighbor_samples}, init_mode={init_mode}\n"
                )
                f.write("-" * 72 + "\n")

                for run_idx in range(num_runs):
                    seed = 5000 + run_idx
                    start_time = time.time()
                    result = run(
                        coords,
                        T_init=T_init,
                        alpha=alpha,
                        T_min=T_min,
                        nr_iter=nr_iter,
                        seed=seed,
                        init_mode=init_mode,
                        neighbor_samples=neighbor_samples,
                    )
                    elapsed = time.time() - start_time

                    best_value = result["best_value"]
                    best_route = result["best_route"]
                    lengths.append(best_value)
                    times.append(elapsed)

                    print(
                        f"    Run {run_idx + 1}/{num_runs}: "
                        f"length={best_value:.2f}, time={elapsed:.2f}s, accepted={result['accepted_moves']}"
                    )
                    f.write(
                        f"Run {run_idx + 1}: length={best_value:.2f}, time={elapsed:.4f}s, "
                        f"accepted={result['accepted_moves']}, route_prefix={best_route[:10]}\n"
                    )

                avg_len = mean(lengths)
                best_len = min(lengths)
                worst_len = max(lengths)
                avg_time = mean(times)
                improvement = 100.0 * (greedy_value - avg_len) / greedy_value

                print(
                    f"    Summary: avg={avg_len:.2f}, best={best_len:.2f}, "
                    f"improvement={improvement:.2f}%, avg_time={avg_time:.2f}s"
                )
                f.write(
                    f">> Avg length={avg_len:.2f} | Best={best_len:.2f} | Worst={worst_len:.2f} | "
                    f"Avg time={avg_time:.4f}s | Avg improvement vs greedy={improvement:.2f}%\n"
                )

                all_rows.append(
                    {
                        "instance": tsp["name"],
                        "n": n,
                        "T_init": T_init,
                        "alpha": alpha,
                        "T_min": T_min,
                        "nr_iter": nr_iter,
                        "neighbor_samples": neighbor_samples,
                        "init_mode": init_mode,
                        "avg_length": avg_len,
                        "best_length": best_len,
                        "avg_time": avg_time,
                        "avg_improvement_pct": improvement,
                    }
                )

    print(f"\nDone! Results saved to {output_filename}")
    return all_rows

In [5]:
assignment_tsp_instances = [
    "../data/A/pr107.tsp",
    "../data/B/lu980.tsp",
]

assignment_parameter_grid = [
    {"T_init": 3000, "alpha": 0.995, "T_min": 1e-4, "nr_iter": 8000, "neighbor_samples": 20, "init_mode": "greedy"},
    {"T_init": 5000, "alpha": 0.994, "T_min": 1e-4, "nr_iter": 12000, "neighbor_samples": 30, "init_mode": "greedy"},
    {"T_init": 8000, "alpha": 0.993, "T_min": 1e-5, "nr_iter": 16000, "neighbor_samples": 40, "init_mode": "random"},
]

sa_results = run_sa_tsp_experiments(
    assignment_tsp_instances,
    assignment_parameter_grid,
    num_runs=3,
    output_filename="results_sa_tsp.txt",
)

print("\nTop settings by average tour length:")
for row in sorted(sa_results, key=lambda x: (x["instance"], x["avg_length"])):
    print(
        f"{row['instance']:>10} | n={row['n']:4d} | T0={row['T_init']:5d} | "
        f"a={row['alpha']:.3f} | iter={row['nr_iter']:5d} | samples={row['neighbor_samples']:2d} | "
        f"init={row['init_mode']:>6} | avg={row['avg_length']:.2f} | best={row['best_length']:.2f} | "
        f"impr={row['avg_improvement_pct']:.2f}% | t={row['avg_time']:.2f}s"
    )


[Instance] pr107 (n=107)
  Greedy baseline: 46678.15
  [Setting 1/3] T_init=3000, alpha=0.995, T_min=0.0001, iter=8000, samples=20, init=greedy
    Run 1/3: length=44824.32, time=0.12s, accepted=701
    Run 2/3: length=45357.33, time=0.14s, accepted=654
    Run 3/3: length=45936.96, time=0.15s, accepted=696
    Summary: avg=45372.87, best=44824.32, improvement=2.80%, avg_time=0.14s
  [Setting 2/3] T_init=5000, alpha=0.994, T_min=0.0001, iter=12000, samples=30, init=greedy
    Run 1/3: length=45065.62, time=0.19s, accepted=652
    Run 2/3: length=45038.95, time=0.16s, accepted=757
    Run 3/3: length=45202.37, time=0.15s, accepted=668
    Summary: avg=45102.31, best=45038.95, improvement=3.38%, avg_time=0.17s
  [Setting 3/3] T_init=8000, alpha=0.993, T_min=1e-05, iter=16000, samples=40, init=random
    Run 1/3: length=46685.08, time=0.22s, accepted=769
    Run 2/3: length=45610.11, time=0.24s, accepted=793
    Run 3/3: length=47052.95, time=0.21s, accepted=758
    Summary: avg=46449.38

## Assignment A4 - Simulated Annealing for Knapsack

### Problem Definition
Given `n` items with `(weight, value)` and capacity `W`, find a 0/1 vector `x` that maximizes:

$$
\max \sum_{i=1}^{n} v_i x_i
$$

subject to:

$$
\sum_{i=1}^{n} w_i x_i \le W, \quad x_i \in \{0,1\}
$$

### Algorithm Used: Simulated Annealing (SA)
We use a one-bit-flip neighborhood for knapsack.

Pseudocode:

```text
current <- random feasible/infeasible binary vector
best <- current
T <- T_init

for k = 1..max_iter:
    if T < T_min: stop
    candidate <- flip one random bit in current

    delta <- f(candidate) - f(current)   # maximization
    if delta >= 0:
        accept candidate
    else accept with probability exp(delta / T)

    if accepted and f(current) > f(best):
        best <- current

    update temperature with one of:
      1) T <- alpha * T
      2) T <- T / log(k + 1)
      3) T <- T / k

    if target value reached: stop

return best
```

We test all 3 temperature functions and compare results on both available instances (`n=20` and `n=200`).

In [6]:
def read_knapsack(filename):
    """Reads instance format used in previous labs."""
    items = []
    capacity = 0

    with open(filename, "r") as f:
        lines = [line.strip() for line in f if line.strip()]

    n = int(lines[0])
    for i in range(1, n + 1):
        parts = lines[i].split()
        # format: index weight value
        weight = int(parts[1])
        value = int(parts[2])
        items.append((weight, value))

    capacity = int(lines[n + 1])
    return items, capacity


def evaluate_knapsack(solution, items, capacity):
    total_weight = 0
    total_value = 0

    for i, bit in enumerate(solution):
        if bit:
            total_weight += items[i][0]
            total_value += items[i][1]

    if total_weight > capacity:
        return 0
    return total_value


def random_solution(n):
    return [random.randint(0, 1) for _ in range(n)]


def random_neighbor_knapsack(solution):
    candidate = solution[:]
    idx = random.randint(0, len(solution) - 1)
    candidate[idx] ^= 1
    return candidate


def random_2flip_indices(n):
    i = random.randint(0, n - 2)
    j = random.randint(i + 1, n - 1)
    return i, j


def random_neighbor_knapsack_2flip(solution):
    candidate = solution[:]
    i, j = random_2flip_indices(len(solution))
    candidate[i] ^= 1
    candidate[j] ^= 1
    return candidate


def random_neighbor_knapsack_mixed(solution):
    if random.random() < 0.5:
        return random_neighbor_knapsack(solution)
    return random_neighbor_knapsack_2flip(solution)


def initial_greedy_knapsack(items, capacity):
    """Value/weight greedy baseline for comparison."""
    n = len(items)
    order = sorted(range(n), key=lambda i: items[i][1] / items[i][0], reverse=True)
    sol = [0] * n
    used = 0

    for i in order:
        w, _ = items[i]
        if used + w <= capacity:
            sol[i] = 1
            used += w

    return sol

In [7]:
def update_temperature(T, k, cooling, alpha):
    """Three required cooling schedules."""
    if cooling == "geom":
        # T(k+1) = alpha * T(k)
        return alpha * T

    if cooling == "log":
        # T(k+1) = T(k) / log(k+1)
        # use k+2 to avoid log(1)=0 at k=0
        denom = math.log(k + 2)
        return T / max(denom, 1e-12)

    if cooling == "invk":
        # T(k+1) = T(k) / k
        # use k+1 in denominator to avoid division by 0
        return T / (k + 1)

    raise ValueError(f"Unknown cooling: {cooling}")


def _build_neighbor(solution, move_mode):
    if move_mode == "1flip":
        return random_neighbor_knapsack(solution)
    if move_mode == "2flip":
        return random_neighbor_knapsack_2flip(solution)
    if move_mode == "mixed":
        return random_neighbor_knapsack_mixed(solution)
    raise ValueError(f"Unknown move_mode: {move_mode}")


def run_sa_knapsack(
    items,
    capacity,
    T_init=10000,
    alpha=0.99,
    T_min=1e-3,
    max_iter=5000,
    cooling="geom",
    target_value=None,
    seed=None,
    init_mode="greedy",
    move_mode="1flip",
    neighbor_samples=1,
):
    """SA for 0/1 Knapsack (maximization)."""
    if seed is not None:
        random.seed(seed)

    n = len(items)
    if init_mode == "greedy":
        current = initial_greedy_knapsack(items, capacity)
    else:
        current = random_solution(n)

    current_value = evaluate_knapsack(current, items, capacity)

    best = current[:]
    best_value = current_value

    T = float(T_init)
    accepted = 0
    history = []

    for k in range(max_iter):
        if T <= T_min:
            break
        if target_value is not None and best_value >= target_value:
            break

        best_candidate = None
        best_candidate_value = -1

        for _ in range(max(1, neighbor_samples)):
            candidate = _build_neighbor(current, move_mode)
            candidate_value = evaluate_knapsack(candidate, items, capacity)
            if candidate_value > best_candidate_value:
                best_candidate = candidate
                best_candidate_value = candidate_value

        delta = best_candidate_value - current_value

        if delta >= 0 or random.random() < math.exp(delta / max(T, 1e-12)):
            current = best_candidate
            current_value = best_candidate_value
            accepted += 1

            if current_value > best_value:
                best = current[:]
                best_value = current_value

        if k % 200 == 0:
            history.append((k, best_value, current_value, T))

        T = update_temperature(T, k, cooling=cooling, alpha=alpha)

    return {
        "best_solution": best,
        "best_value": best_value,
        "accepted_moves": accepted,
        "history": history,
    }

In [8]:
def run_sa_knapsack_experiments(instance_file, parameter_grid, num_runs=5):
    items, capacity = read_knapsack(instance_file)
    n = len(items)
    output_file = f"results_sa_knapsack_n{n}.txt"

    greedy_sol = initial_greedy_knapsack(items, capacity)
    greedy_value = evaluate_knapsack(greedy_sol, items, capacity)

    rows = []

    with open(output_file, "w") as f:
        f.write(f"--- Simulated Annealing Knapsack Results for {instance_file} ---\n")
        f.write(f"n={n}, capacity={capacity}, runs_per_setting={num_runs}\n")
        f.write(f"Greedy baseline value={greedy_value}\n")

        for idx, params in enumerate(parameter_grid, start=1):
            T_init = params["T_init"]
            alpha = params["alpha"]
            T_min = params["T_min"]
            max_iter = params["max_iter"]
            cooling = params["cooling"]
            init_mode = params.get("init_mode", "greedy")
            move_mode = params.get("move_mode", "1flip")
            neighbor_samples = params.get("neighbor_samples", 1)
            target_value = params.get("target_value")

            values = []
            times = []

            print(
                f"[n={n}] setting {idx}/{len(parameter_grid)}: "
                f"cool={cooling}, init={init_mode}, move={move_mode}, samples={neighbor_samples}, "
                f"T0={T_init}, alpha={alpha}, Tmin={T_min}, iter={max_iter}"
            )

            f.write("\n" + "=" * 70 + "\n")
            f.write(
                f"Setting {idx}: cooling={cooling}, init_mode={init_mode}, move_mode={move_mode}, "
                f"neighbor_samples={neighbor_samples}, T_init={T_init}, alpha={alpha}, "
                f"T_min={T_min}, max_iter={max_iter}, target={target_value}\n"
            )
            f.write("-" * 70 + "\n")

            for run_idx in range(num_runs):
                seed = 7000 + run_idx
                t0 = time.time()
                result = run_sa_knapsack(
                    items,
                    capacity,
                    T_init=T_init,
                    alpha=alpha,
                    T_min=T_min,
                    max_iter=max_iter,
                    cooling=cooling,
                    target_value=target_value,
                    seed=seed,
                    init_mode=init_mode,
                    move_mode=move_mode,
                    neighbor_samples=neighbor_samples,
                )
                elapsed = time.time() - t0

                values.append(result["best_value"])
                times.append(elapsed)

                f.write(
                    f"Run {run_idx + 1}: value={result['best_value']}, "
                    f"time={elapsed:.4f}s, accepted={result['accepted_moves']}\n"
                )

            avg_val = mean(values)
            best_val = max(values)
            worst_val = min(values)
            avg_time = mean(times)
            improve = 100.0 * (avg_val - greedy_value) / max(greedy_value, 1)

            f.write(
                f">> avg={avg_val:.2f}, best={best_val}, worst={worst_val}, "
                f"avg_time={avg_time:.4f}s, avg_improvement_vs_greedy={improve:.2f}%\n"
            )

            rows.append(
                {
                    "n": n,
                    "cooling": cooling,
                    "init_mode": init_mode,
                    "move_mode": move_mode,
                    "neighbor_samples": neighbor_samples,
                    "T_init": T_init,
                    "alpha": alpha,
                    "T_min": T_min,
                    "max_iter": max_iter,
                    "avg": avg_val,
                    "best": best_val,
                    "worst": worst_val,
                    "avg_time": avg_time,
                    "avg_improvement_pct": improve,
                }
            )

    print(f"Done: {output_file}")
    return rows

In [9]:
knapsack_20_grid = [
    {"cooling": "geom", "T_init": 10000, "alpha": 0.99,  "T_min": 1e-3, "max_iter": 3000, "init_mode": "greedy", "move_mode": "1flip", "neighbor_samples": 1},
    {"cooling": "geom", "T_init": 10000, "alpha": 0.995, "T_min": 1e-3, "max_iter": 5000, "init_mode": "greedy", "move_mode": "2flip", "neighbor_samples": 25},
    {"cooling": "log",  "T_init": 10000, "alpha": 0.99,  "T_min": 1e-3, "max_iter": 3000, "init_mode": "greedy", "move_mode": "mixed", "neighbor_samples": 20},
    {"cooling": "invk", "T_init": 10000, "alpha": 0.99,  "T_min": 1e-3, "max_iter": 3000, "init_mode": "greedy", "move_mode": "mixed", "neighbor_samples": 20},
]

knapsack_200_grid = [
    {"cooling": "geom", "T_init": 10000, "alpha": 0.99,  "T_min": 1e-3, "max_iter": 6000, "init_mode": "greedy", "move_mode": "1flip", "neighbor_samples": 1},
    {"cooling": "geom", "T_init": 10000, "alpha": 0.995, "T_min": 1e-3, "max_iter": 9000, "init_mode": "greedy", "move_mode": "2flip", "neighbor_samples": 40},
    {"cooling": "log",  "T_init": 10000, "alpha": 0.99,  "T_min": 1e-3, "max_iter": 6000, "init_mode": "greedy", "move_mode": "mixed", "neighbor_samples": 30},
    {"cooling": "invk", "T_init": 10000, "alpha": 0.99,  "T_min": 1e-3, "max_iter": 6000, "init_mode": "greedy", "move_mode": "mixed", "neighbor_samples": 30},
]

rows20 = run_sa_knapsack_experiments("../data/knapsack-20.txt", knapsack_20_grid, num_runs=5)
rows200 = run_sa_knapsack_experiments("../data/knapsack-200.txt", knapsack_200_grid, num_runs=5)

all_rows = rows20 + rows200
print("\nSummary by instance (sorted by average value desc):")
for n in [20, 200]:
    subset = [r for r in all_rows if r["n"] == n]
    subset = sorted(subset, key=lambda r: r["avg"], reverse=True)
    print(f"\nInstance n={n}")
    for r in subset:
        print(
            f"cool={r['cooling']:>4} | init={r['init_mode']:>6} | move={r['move_mode']:>5} | "
            f"samples={r['neighbor_samples']:2d} | T0={r['T_init']:5d} | a={r['alpha']:.3f} | "
            f"iter={r['max_iter']:5d} | avg={r['avg']:.2f} | best={r['best']} | "
            f"impr={r['avg_improvement_pct']:.2f}% | t={r['avg_time']:.3f}s"
        )

[n=20] setting 1/4: cool=geom, init=greedy, move=1flip, samples=1, T0=10000, alpha=0.99, Tmin=0.001, iter=3000
[n=20] setting 2/4: cool=geom, init=greedy, move=2flip, samples=25, T0=10000, alpha=0.995, Tmin=0.001, iter=5000
[n=20] setting 3/4: cool=log, init=greedy, move=mixed, samples=20, T0=10000, alpha=0.99, Tmin=0.001, iter=3000
[n=20] setting 4/4: cool=invk, init=greedy, move=mixed, samples=20, T0=10000, alpha=0.99, Tmin=0.001, iter=3000
Done: results_sa_knapsack_n20.txt
[n=200] setting 1/4: cool=geom, init=greedy, move=1flip, samples=1, T0=10000, alpha=0.99, Tmin=0.001, iter=6000
[n=200] setting 2/4: cool=geom, init=greedy, move=2flip, samples=40, T0=10000, alpha=0.995, Tmin=0.001, iter=9000
[n=200] setting 3/4: cool=log, init=greedy, move=mixed, samples=30, T0=10000, alpha=0.99, Tmin=0.001, iter=6000
[n=200] setting 4/4: cool=invk, init=greedy, move=mixed, samples=30, T0=10000, alpha=0.99, Tmin=0.001, iter=6000
Done: results_sa_knapsack_n200.txt

Summary by instance (sorted by a

## A4 Report - Simulated Annealing for Knapsack

### 1) Problem Definition
We solve the 0/1 Knapsack problem.
- Each item has a weight and a value.
- The backpack has capacity `W`.
- We choose items to maximize total value without passing capacity.

Mathematically:

$$
\max \sum_{i=1}^{n} v_i x_i
$$

subject to

$$
\sum_{i=1}^{n} w_i x_i \le W, \quad x_i \in \{0,1\}
$$

### 2) Algorithm Used
Algorithm: Simulated Annealing (SA), maximization form.

Main steps:
1. Start from an initial solution (`greedy` in our runs).
2. Build neighbor solutions by bit flips:
   - `1flip` (flip one bit)
   - `2flip` (flip two bits)
   - `mixed` (randomly choose 1flip or 2flip)
3. Evaluate candidate value.
4. Accept always if better.
5. If worse, accept with probability $e^{\Delta/T}$ where $\Delta = f(candidate)-f(current)$.
6. Update temperature with one cooling schedule.
7. Stop when temperature is below `T_min`, max iterations is reached, or target value is reached.



### 3) Comparative Results
From the experiment files:
- For `n=20`, greedy baseline value is `787`.
- For `n=200`, greedy baseline value is `101068`.
- For all parameter settings and all cooling schedules, SA reached the same best/average value as greedy.
- So average improvement vs greedy is `0.00%` for all tested settings.

### 4) Why Improvement Is 0%
We also checked exact optimum using dynamic programming.
- `n=20`: greedy = `787`, optimum = `787`
- `n=200`: greedy = `101068`, optimum = `101068`

Conclusion: on these two instances, greedy is already optimal, so SA cannot produce a higher value.

### 5) Discussion
- Geometric cooling accepted more moves and took more time.
- Log and inverse-k cooled faster, accepted fewer moves, and ran faster.
- Even with stronger neighborhoods (`2flip`, `mixed`) and more samples, final quality stayed the same because baseline is already optimal.
- The assignment requirements are met: documented source code, multiple settings, all instances, saved outputs, and comparative discussion.